# 30 — Validate `usdm_v4.ttl`

Parses the generated Turtle, runs SPARQL sanity counts, cross-checks NCIt
anchors against the source YAML, and appends a CSV row to
`../reports/validation.csv`.

Comparison against the prototype baseline (8,215 triples) is informational.
Deviation flags either source drift (likely benign, document in `docs/`) or
a generation bug (investigate).


## 1. Parse `usdm_v4.ttl`

Fail fast on syntax errors. Successful parse ⇒ the file is valid Turtle.


In [1]:
import rdflib
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/20_generate_turtle.ipynb first."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
print(f"parsed {len(g)} triples from {TTL_PATH}")


parsed 8642 triples from ../usdm_v4.ttl


## 2. Load source YAML for cross-checks

We re-parse `dataStructure.yml` here to compare what's in the graph
against what's in source.


In [2]:
import yaml
import json

YAML_PATH = "../downloads/dataStructure.yml"
META_PATH = "../downloads/.fetch_meta.json"

with open(YAML_PATH) as f:
    SOURCE = yaml.safe_load(f)
with open(META_PATH) as f:
    META = json.load(f)

print(f"YAML  : {len(SOURCE)} top-level entries")
print(f"tag   : {META['ddf_ra_tag']}")
print(f"sha256: {META['sha256']}")


YAML  : 86 top-level entries
tag   : v4.0.0
sha256: 6f49a407aef41a66dd40a82ad3348672a22cd8874849254d7e4f27d6275daca6


## 2b. Load USDM_CT bindings (for cross-checks)

Same loader as `20_generate_turtle.ipynb`. Builds `CT_BINDINGS` keyed by
`(entity, attr_name)` for verifying that every Y-row in source resolves
to the expected annotation triples in the graph.
Rows where `Inherited From` is non-null are skipped — the property IRI
exists only on the declaring class per v0's mechanical mapping, and the
inherited row duplicates the parent's binding.


In [3]:
import openpyxl
import re
from pathlib import Path

CT_PATH      = "../downloads/USDM_CT.xlsx"
CT_META_PATH = "../downloads/.fetch_meta_ct.json"
if not Path(CT_PATH).exists():
    raise FileNotFoundError(
        f"{CT_PATH} missing — run notebooks/10_fetch_yaml.ipynb first."
    )

with open(CT_META_PATH) as f:
    CT_META = json.load(f)

CODE_TYPED_TARGETS = {"Code", "AliasCode", "ResponseCode"}
RE_URL_CCODE  = re.compile(r"/subset/ncit/(C\d+)\s*$")
RE_TEXT_CCODE = re.compile(r"\b(C\d+)\b")

wb = openpyxl.load_workbook(CT_PATH, read_only=True, data_only=True)
ws = wb["DDF Entities&Attributes"]
rows = list(ws.iter_rows(values_only=True))
header = list(rows[0])
i_entity = header.index("Entity Name")
i_attr   = header.index("Logical Data Model Name")
i_hvl    = header.index("Has Value List")
i_url    = header.index("Codelist URL")
i_inh    = header.index("Inherited From")

CT_BINDINGS = {}
for r in rows[1:]:
    if r[i_inh]:
        continue
    hvl = r[i_hvl]
    if not hvl or not str(hvl).startswith("Y"):
        continue
    ent = r[i_entity]
    att = r[i_attr]
    url = r[i_url]
    ccode_url, ccode_text = None, None
    if url:
        m = RE_URL_CCODE.search(str(url))
        if m: ccode_url = m.group(1)
    text_codes = RE_TEXT_CCODE.findall(str(hvl))
    if len(text_codes) > 1:
        raise ValueError(f"multiple C-codes in HVL for {ent}.{att}: {hvl!r}")
    if text_codes:
        ccode_text = text_codes[0]
    if ccode_url and ccode_text and ccode_url != ccode_text:
        raise ValueError(
            f"C-code mismatch for {ent}.{att}: url={ccode_url} text={ccode_text} hvl={hvl!r}"
        )
    ccode = ccode_url or ccode_text
    CT_BINDINGS[(ent, att)] = {
        "raw_hvl": str(hvl),
        "ccode":   ccode,
    }

n_total      = len(CT_BINDINGS)
n_structured = sum(1 for v in CT_BINDINGS.values() if v["ccode"])
n_freetext   = n_total - n_structured
print(f"CT_BINDINGS:   {n_total} Y-rows ({n_structured} structured, {n_freetext} free-text)")
print(f"ct tag:        {CT_META['ddf_ra_tag']}")
print(f"ct sha256:     {CT_META['sha256']}")


CT_BINDINGS:   57 Y-rows (45 structured, 12 free-text)
ct tag:        v4.0.0
ct sha256:     a9efa740abc41299efa83607a0880012ac8dcc8df7db60259c2a7e96f42cd94f


## 3. SPARQL sanity counts

Named classes (filter to IRIs to exclude `owl:unionOf` blank nodes).
Properties = `owl:DatatypeProperty` ∪ `owl:ObjectProperty`.


In [4]:
QUERIES = {
    "triples": (
        "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"
    ),
    "classes_total": (
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a owl:Class . FILTER(isIRI(?c)) }"
    ),
    "classes_nci_anchored": (
        "PREFIX skos:<http://www.w3.org/2004/02/skos/core#> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { "
        "  ?c a owl:Class ; skos:exactMatch ?x . FILTER(isIRI(?c)) }"
    ),
    "properties_total": (
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { "
        "  { ?p a owl:DatatypeProperty } UNION { ?p a owl:ObjectProperty } }"
    ),
    "properties_nci_anchored": (
        "PREFIX skos:<http://www.w3.org/2004/02/skos/core#> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { "
        "  ?p skos:exactMatch ?x . "
        "  FILTER EXISTS { { ?p a owl:DatatypeProperty } UNION { ?p a owl:ObjectProperty } } }"
    ),
    "value_props": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p usdm:relationshipType 'Value' }"
    ),
    "ref_props": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p usdm:relationshipType 'Ref' }"
    ),
    "concrete_classes": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c usdm:modifier 'Concrete' . FILTER(isIRI(?c)) }"
    ),
    "abstract_classes": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c usdm:modifier 'Abstract' . FILTER(isIRI(?c)) }"
    ),
    "boundCodelist_triples": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(*) AS ?n) WHERE { ?p usdm:boundCodelist ?x }"
    ),
    "boundCodelistNote_triples": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(*) AS ?n) WHERE { ?p usdm:boundCodelistNote ?x }"
    ),
    "boundCodelist_nonNcit": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "PREFIX ncit:<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#> "
        "SELECT (COUNT(*) AS ?n) WHERE { "
        "  ?p usdm:boundCodelist ?x . "
        "  FILTER(!STRSTARTS(STR(?x), STR(ncit:)) && "
        "         !STRSTARTS(STR(?x), 'http://purl.obolibrary.org/obo/NCIT_')) }"
    ),
}

counts = {}
for label, sparql in QUERIES.items():
    counts[label] = int(list(g.query(sparql))[0][0])
    print(f"  {label:<25}: {counts[label]}")


  triples                  : 8642
  classes_total            : 86
  classes_nci_anchored     : 84
  properties_total         : 693
  properties_nci_anchored  : 312
  value_props              : 630
  ref_props                : 63
  concrete_classes         : 80
  abstract_classes         : 6
  boundCodelist_triples    : 90
  boundCodelistNote_triples: 57
  boundCodelist_nonNcit    : 0


## 4. Cross-check class-level NCIt anchors

For every source class with `NCI C-Code`, the graph must contain
`usdm:{Class} skos:exactMatch ncit:{C-Code}`. Mismatches are collected,
not raised — we want a complete report, not a first-failure exit.


In [5]:
USDM = rdflib.Namespace("https://w3id.org/cdisc/usdm/v4/")
NCIT = rdflib.Namespace("http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#")
OBO  = rdflib.Namespace("http://purl.obolibrary.org/obo/")
SKOS = rdflib.Namespace("http://www.w3.org/2004/02/skos/core#")

class_mismatches = []
for cname, entry in SOURCE.items():
    if not isinstance(entry, dict) or "NCI C-Code" not in entry:
        continue
    expected = NCIT[entry["NCI C-Code"]]
    subject = USDM[cname]
    if (subject, SKOS.exactMatch, expected) not in g:
        class_mismatches.append((cname, entry["NCI C-Code"]))
    expected_obo = OBO[f"NCIT_{entry['NCI C-Code']}"]
    if (subject, SKOS.exactMatch, expected_obo) not in g:
        class_mismatches.append((cname, f"OBO twin {entry['NCI C-Code']}"))

print(f"class NCIt cross-check: {len(class_mismatches)} mismatches")
for c, code in class_mismatches[:20]:
    print(f"  MISSING: usdm:{c} skos:exactMatch ncit:{code}")


class NCIt cross-check: 0 mismatches


## 5. Cross-check attribute-level NCIt anchors

Same check, scoped to non-inherited attributes. Property IRI is
`usdm:{ClassName}-{attributeName}`.


In [6]:
attr_mismatches = []
for cname, entry in SOURCE.items():
    if not isinstance(entry, dict):
        continue
    for aname, attr in (entry.get("Attributes") or {}).items():
        if not isinstance(attr, dict) or "Inherited From" in attr:
            continue
        if "NCI C-Code" not in attr:
            continue
        expected = NCIT[attr["NCI C-Code"]]
        subject = USDM[f"{cname}-{aname}"]
        if (subject, SKOS.exactMatch, expected) not in g:
            attr_mismatches.append((cname, aname, attr["NCI C-Code"]))
        expected_obo = OBO[f"NCIT_{attr['NCI C-Code']}"]
        if (subject, SKOS.exactMatch, expected_obo) not in g:
            attr_mismatches.append((cname, aname, f"OBO twin {attr['NCI C-Code']}"))

print(f"attribute NCIt cross-check: {len(attr_mismatches)} mismatches")
for c, a, code in attr_mismatches[:20]:
    print(f"  MISSING: usdm:{c}-{a} skos:exactMatch ncit:{code}")


attribute NCIt cross-check: 0 mismatches


## 5b. Cross-check codelist bindings

For every Y-row in `CT_BINDINGS`, the graph must contain
`usdm:{Class}-{attr} usdm:boundCodelistNote "<raw>"` and, when a
structured codelist C-code is present, also
`usdm:{Class}-{attr} usdm:boundCodelist ncit:Cxxxxxx`.


In [7]:
USDM_BOUND_CODELIST      = USDM["boundCodelist"]
USDM_BOUND_CODELIST_NOTE = USDM["boundCodelistNote"]

binding_mismatches = []
for (cname, aname), b in CT_BINDINGS.items():
    subject = USDM[f"{cname}-{aname}"]
    expected_note = rdflib.Literal(b["raw_hvl"])
    if (subject, USDM_BOUND_CODELIST_NOTE, expected_note) not in g:
        binding_mismatches.append((cname, aname, "boundCodelistNote", b["raw_hvl"]))
    if b["ccode"]:
        expected_ccode = NCIT[b["ccode"]]
        if (subject, USDM_BOUND_CODELIST, expected_ccode) not in g:
            binding_mismatches.append((cname, aname, "boundCodelist", b["ccode"]))
        expected_obo = OBO[f"NCIT_{b['ccode']}"]
        if (subject, USDM_BOUND_CODELIST, expected_obo) not in g:
            binding_mismatches.append((cname, aname, "boundCodelist", f"OBO twin {b['ccode']}"))

print(f"binding cross-check: {len(binding_mismatches)} mismatches")
for c, a, kind, val in binding_mismatches[:20]:
    print(f"  MISSING: usdm:{c}-{a} usdm:{kind} {val!r}")


binding cross-check: 0 mismatches


## 6. Compare against prototype baseline

Baseline numbers come from prior prototype work against an earlier DDF-RA
snapshot. The triple-count baseline (8,215) reflects choices in that
prototype we cannot fully reconstruct — a residual delta of <300 is
expected and documented in `docs/known-gaps.md`. Structural counts
(classes, properties, anchored counts) should match exactly for tag
`v4.0.0` of DDF-RA.


In [8]:
BASELINE = {
    "triples":                   8642,
    "classes_total":               86,
    "classes_nci_anchored":        84,
    "properties_total":           693,
    "properties_nci_anchored":    312,
    "boundCodelist_triples":       90,
    "boundCodelistNote_triples":   57,
}

print(f"{'metric':<25} {'baseline':>10} {'observed':>10} {'delta':>10}")
print("-" * 60)
for k, base in BASELINE.items():
    obs = counts[k]
    delta = obs - base
    print(f"{k:<25} {base:>10} {obs:>10} {delta:>+10}")


metric                      baseline   observed      delta
------------------------------------------------------------
triples                         8642       8642         +0
classes_total                     86         86         +0
classes_nci_anchored              84         84         +0
properties_total                 693        693         +0
properties_nci_anchored          312        312         +0
boundCodelist_triples             90         90         +0
boundCodelistNote_triples         57         57         +0


## 7. Append CSV row to `../reports/validation.csv`

One row per validation run, header on first write. History accumulates
in the file across runs.


In [9]:
import csv
import datetime
from pathlib import Path

REPORT_PATH = "../reports/validation.csv"

row = {
    "timestamp_utc":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "ddf_ra_tag":              META["ddf_ra_tag"],
    "source_sha256":           META["sha256"],
    "triples":                 counts["triples"],
    "classes_total":           counts["classes_total"],
    "classes_nci_anchored":    counts["classes_nci_anchored"],
    "properties_total":        counts["properties_total"],
    "properties_nci_anchored": counts["properties_nci_anchored"],
    "value_props":             counts["value_props"],
    "ref_props":               counts["ref_props"],
    "concrete_classes":        counts["concrete_classes"],
    "abstract_classes":        counts["abstract_classes"],
    "class_mismatches":          len(class_mismatches),
    "attr_mismatches":           len(attr_mismatches),
    "ct_source_tag":             CT_META["ddf_ra_tag"],
    "ct_source_sha256":          CT_META["sha256"],
    "boundCodelist_triples":     counts["boundCodelist_triples"],
    "boundCodelistNote_triples": counts["boundCodelistNote_triples"],
    "binding_mismatches":        len(binding_mismatches),
}

Path(REPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
first_write = (not Path(REPORT_PATH).exists()) or Path(REPORT_PATH).stat().st_size == 0
with open(REPORT_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    if first_write:
        w.writeheader()
    w.writerow(row)

action = "created" if first_write else "appended to"
print(f"{action} {REPORT_PATH}")
for k, v in row.items():
    print(f"  {k:<25}: {v}")


appended to ../reports/validation.csv
  timestamp_utc            : 2026-07-03T18:03:12.109828+00:00
  ddf_ra_tag               : v4.0.0
  source_sha256            : 6f49a407aef41a66dd40a82ad3348672a22cd8874849254d7e4f27d6275daca6
  triples                  : 8642
  classes_total            : 86
  classes_nci_anchored     : 84
  properties_total         : 693
  properties_nci_anchored  : 312
  value_props              : 630
  ref_props                : 63
  concrete_classes         : 80
  abstract_classes         : 6
  class_mismatches         : 0
  attr_mismatches          : 0
  ct_source_tag            : v4.0.0
  ct_source_sha256         : a9efa740abc41299efa83607a0880012ac8dcc8df7db60259c2a7e96f42cd94f
  boundCodelist_triples    : 90
  boundCodelistNote_triples: 57
  binding_mismatches       : 0


## 8. Run summary

Empty mismatch lists ⇒ every NCI C-Code in source has a matching
`skos:exactMatch` triple in the graph. Non-empty lists are real bugs
to investigate.


In [10]:
n_class_mm = len(class_mismatches)
n_attr_mm  = len(attr_mismatches)
n_bind_mm  = len(binding_mismatches)

if n_class_mm == 0 and n_attr_mm == 0 and n_bind_mm == 0:
    print("clean run: all source NCI anchors and codelist bindings present in graph")
else:
    print(f"class mismatches:     {n_class_mm}")
    print(f"attribute mismatches: {n_attr_mm}")
    print(f"binding mismatches:   {n_bind_mm}")
    print("see cells 4, 5, and 5b for details")


clean run: all source NCI anchors and codelist bindings present in graph


## 9. JSON-LD instance context check

Expand the CDISC Pilot example study (real instance data published in
DDF-RA at the pinned tag,
`Documents/Examples/CDISC_Pilot/CDISC_Pilot_Study.json`) with
`../usdm_v4.context.jsonld` and check the resulting graph against the
ontology. The example file is test data, not a modelling source, so it
is fetched here rather than in `10_fetch_yaml.ipynb` — that notebook
holds sources of truth only. SHA-256 sidecar: `.fetch_meta_pilot.json`.

Checks (baselines for DDF-RA v4.0.0): every predicate is a declared
property and every `rdf:type` a declared class; typed nodes = 1,953 (one
per typed JSON object); no unmapped serialization keys; all 1,257 Ref
links resolve to typed nodes, 0 dangling; 1 known source quirk — the
root `Study` object has `id: null` and lands as a blank node. Results
append to `../reports/context_validation.csv`.


In [11]:
import csv
import hashlib
import json
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import rdflib
from rdflib.namespace import OWL, RDF

CONTEXT_PATH = "../usdm_v4.context.jsonld"
PILOT_PATH = "../downloads/CDISC_Pilot_Study.json"
PILOT_META_PATH = "../downloads/.fetch_meta_pilot.json"

if not Path(CONTEXT_PATH).exists():
    raise FileNotFoundError(
        f"{CONTEXT_PATH} missing — run notebooks/40_generate_context.ipynb first."
    )

with open("../downloads/.fetch_meta.json") as f:
    DDF_RA_TAG = json.load(f)["ddf_ra_tag"]
PILOT_URL = (
    f"https://raw.githubusercontent.com/cdisc-org/DDF-RA/{DDF_RA_TAG}"
    "/Documents/Examples/CDISC_Pilot/CDISC_Pilot_Study.json"
)

if not Path(PILOT_PATH).exists():
    with urllib.request.urlopen(PILOT_URL) as resp:
        data = resp.read()
    with open(PILOT_PATH, "wb") as f:
        f.write(data)
    with open(PILOT_META_PATH, "w") as f:
        json.dump({
            "ddf_ra_tag": DDF_RA_TAG,
            "raw_url": PILOT_URL,
            "sha256": hashlib.sha256(data).hexdigest(),
            "size_bytes": len(data),
            "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
        }, f, indent=2)
        f.write("\n")

with open(CONTEXT_PATH) as f:
    context = json.load(f)["@context"]
with open(PILOT_PATH) as f:
    doc = json.load(f)

assert "study" in doc, "pilot JSON lacks 'study' envelope key"
envelope_keys = sorted(k for k in doc if k != "study")
study = doc["study"]

unmapped = {}
null_ids = 0

def walk(node):
    global null_ids
    if isinstance(node, dict):
        instance_type = node.get("instanceType")
        if isinstance(instance_type, str):
            scoped = context[instance_type]["@context"]
            for k in node:
                if k not in scoped and k not in ("id", "instanceType"):
                    unmapped.setdefault(instance_type, set()).add(k)
            if node.get("id") is None:
                null_ids += 1
        for k, v in node.items():
            if k != "@context":
                walk(v)
    elif isinstance(node, list):
        for v in node:
            walk(v)

walk(study)
assert not unmapped, f"unmapped serialization keys: {unmapped}"

study["@context"] = context
instances = rdflib.Graph()
instances.parse(data=json.dumps(study), format="json-ld", publicID=PILOT_URL)

ont = rdflib.Graph()
ont.parse(TTL_PATH, format="turtle")
declared_props = (
    set(ont.subjects(RDF.type, OWL.DatatypeProperty))
    | set(ont.subjects(RDF.type, OWL.ObjectProperty))
)
declared_classes = {c for c in ont.subjects(RDF.type, OWL.Class) if isinstance(c, rdflib.URIRef)}

bad_preds = {p for p in set(instances.predicates()) if p != RDF.type and p not in declared_props}
assert not bad_preds, f"undeclared predicates: {sorted(bad_preds)[:5]}"
bad_types = {t for t in set(instances.objects(None, RDF.type)) if t not in declared_classes}
assert not bad_types, f"undeclared types: {sorted(bad_types)[:5]}"

ref_props = set()
for class_def in context.values():
    if isinstance(class_def, dict) and "@context" in class_def:
        for term in class_def["@context"].values():
            if term.get("@type") == "@id":
                ref_props.add(rdflib.URIRef(
                    term["@id"].replace("usdm:", str(USDM))
                ))

ref_links = 0
dangling = 0
for p in ref_props:
    for s, o in instances.subject_objects(p):
        ref_links += 1
        if (o, RDF.type, None) not in instances:
            dangling += 1

typed_nodes = len(set(instances.subjects(RDF.type, None)))

CONTEXT_BASELINE = {
    "instance_triples": 11811,
    "typed_nodes": 1953,
    "ref_links": 1257,
    "dangling_ref_links": 0,
    "null_ids": 1,
}
observed = {
    "instance_triples": len(instances),
    "typed_nodes": typed_nodes,
    "ref_links": ref_links,
    "dangling_ref_links": dangling,
    "null_ids": null_ids,
}
for k, expected in CONTEXT_BASELINE.items():
    assert observed[k] == expected, f"{k}: expected {expected}, got {observed[k]}"

report_path = Path("../reports/context_validation.csv")
write_header = not report_path.exists()
with open(report_path, "a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["run_at_utc", "ddf_ra_tag", "envelope_keys"] + list(observed))
    writer.writerow(
        [datetime.now(timezone.utc).isoformat(), DDF_RA_TAG, " ".join(envelope_keys)]
        + [observed[k] for k in observed]
    )

print("envelope keys (out of model, dropped):", envelope_keys)
for k, v in observed.items():
    print(f"{k:<20} {v:>8}")
print("context check: PASS")


envelope keys (out of model, dropped): ['systemName', 'systemVersion', 'usdmVersion']
instance_triples        11811
typed_nodes              1953
ref_links                1257
dangling_ref_links          0
null_ids                    1
context check: PASS


## Post-deploy: live representation consistency

The five content-negotiated representations at the namespace IRI must
be the same graph. This cell fetches the four RDF formats live,
normalizes a known N-Triples serializer artifact (explicit
`rdf:type rdf:List` on `owl:unionOf` list cells), and asserts equal
version markers, equal ground-triple sets, and equal triple counts;
it also checks the HTML representation dereferences with per-entity
anchors.

Run this after tagging a release and merging the w3id re-pin. It is
expected to fail in the window between local regeneration and release
completion — that failure is the guard working, not a pipeline bug.

In [12]:
# Post-deploy representation-consistency check (F1 regression guard).
# Compares the live content-negotiated representations of the namespace
# IRI against each other. Run after a release is tagged and the w3id
# re-pin is merged. Expected to FAIL between regenerating locally and
# completing the release — the live state lags this repo by design.

import urllib.request

import rdflib
from rdflib import RDF, RDFS, OWL, URIRef
from rdflib.namespace import DCTERMS, XSD

NAMESPACE = "https://w3id.org/cdisc/usdm/v4/"
ONTO = URIRef(NAMESPACE)

RDF_REPRESENTATIONS = {
    "turtle":   ("text/turtle", "turtle"),
    "jsonld":   ("application/ld+json", "json-ld"),
    "rdfxml":   ("application/rdf+xml", "xml"),
    "ntriples": ("application/n-triples", "nt"),
}

graphs = {}
for name, (accept, fmt) in RDF_REPRESENTATIONS.items():
    req = urllib.request.Request(NAMESPACE, headers={"Accept": accept, "User-Agent": "usdm-rdf-validate"})
    with urllib.request.urlopen(req) as resp:
        data = resp.read()
    g = rdflib.Graph()
    g.parse(data=data, format=fmt)
    # Normalize: drop explicit rdf:type rdf:List triples — an OWLAPI
    # serializer artifact on owl:unionOf list cells (11 in the
    # N-Triples output, 0 elsewhere). Semantically redundant.
    for s in list(g.subjects(RDF.type, RDF.List)):
        g.remove((s, RDF.type, RDF.List))
    # Same class of artifact: OWLAPI declares used XSD datatypes
    # (xsd:date a rdfs:Datatype). Built-in datatypes need no
    # declaration in OWL 2; drop declarations in the XSD namespace.
    for s in list(g.subjects(RDF.type, RDFS.Datatype)):
        if str(s).startswith(str(XSD)):
            g.remove((s, RDF.type, RDFS.Datatype))
    graphs[name] = g

# 1. Version markers identical across RDF representations.
markers = {}
for name, g in graphs.items():
    markers[name] = (
        sorted(map(str, g.objects(ONTO, OWL.versionInfo))),
        sorted(map(str, g.objects(ONTO, OWL.versionIRI))),
        sorted(map(str, g.objects(ONTO, DCTERMS.created))),
        sorted(map(str, g.objects(ONTO, DCTERMS.modified))),
    )
reference = markers["turtle"]
for name, m in markers.items():
    assert m == reference, f"version markers diverge: turtle={reference} vs {name}={m}"

# 2. Ground (bnode-free) triple sets identical.
def ground_triples(g):
    return {t for t in g if not any(isinstance(x, rdflib.BNode) for x in t)}

ground_reference = ground_triples(graphs["turtle"])
for name, g in graphs.items():
    delta = ground_triples(g) ^ ground_reference
    assert not delta, f"{name}: {len(delta)} ground-triple differences vs turtle"

# 3. Total triple counts identical after normalization. owl:unionOf
#    list *order* may differ between serializers; equal counts plus
#    equal ground sets bound the bnode structures without requiring
#    full isomorphism.
counts = {name: len(g) for name, g in graphs.items()}
assert len(set(counts.values())) == 1, f"triple counts diverge: {counts}"

# 4. HTML representation reachable with per-entity anchors.
req = urllib.request.Request(NAMESPACE + "Activity", headers={"Accept": "text/html", "User-Agent": "Mozilla/5.0 usdm-rdf-validate"})
with urllib.request.urlopen(req) as resp:
    assert resp.status == 200, f"HTML dereference returned {resp.status}"
    html = resp.read().decode("utf-8")
assert 'id="Activity"' in html, "HTML page lacks the Activity anchor"

print("Live representations consistent:", counts)
print("Version markers:", reference)

Live representations consistent: {'turtle': 8641, 'jsonld': 8641, 'rdfxml': 8641, 'ntriples': 8641}
Version markers: (['v0.4.0'], ['https://w3id.org/cdisc/usdm/v4/0.4.0'], ['2026-04-27'], ['2026-06-12'])
